# 年报文本清洗分析

## 1. 数据探索部分

通过对年报文件的初步分析，发现以下问题：

1. **标记语言标签**：文件中包含大量HTML标签，如 `<html>`, `<body>`, `<div>` 等，这些标签对文本分析无意义
2. **控制和特殊字符**：存在ASCII控制字符（0-31和127）、制表符、多个连续空格、换行符等排版问题
3. **系统生成内容**：
   - 技术元数据：如Proc-Type、Originator-Name等系统生成的元数据
   - 加密字符串：如RSA密钥、MD5哈希等长字符串
   - 文件名和扩展名：如19970321_10--97-000002_1、.htm、.html、.txt等系统生成的文件名和扩展名
4. **无意义字段**：比较典型的是"table of contents"，在某几份研报中反复出现。

In [9]:
import re
import os

## 2. 代码实现部分

In [10]:
def clean_annual_report(text):
    # 1. 处理 HTML 标签：保留标签内文本>=500 字符的内容
    html_pattern = r'<[^>]*>.*?</[^>]+>'
    html_matches = re.findall(html_pattern, text, flags=re.S)
    
    for tag in html_matches:
        # 提取标签内的文本内容
        text_content = re.sub(r'<[^>]+>', '', tag)
        # 根据长度决定保留或删除
        if len(text_content) >= 500:
            continue  # 保留，不做处理
        else:
            text = text.replace(tag, '')  # 删除
    
    # 2. 移除剩余的孤立 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. 移除控制和特殊字符
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    
    # 4. 删除"table of contents"（忽略大小写）
    text = re.sub(r'\btable\s+of\s+contents\b', '', text, flags=re.I)
    
    # 5. 删除连续出现3次及以上的特殊字符（., -, = 等）
    text = re.sub(r'[.]{3,}', '', text)
    text = re.sub(r'[-]{3,}', '', text)
    text = re.sub(r'[=]{3,}', '', text)
    
    # 6. 移除系统生成内容
    text = re.sub(r'Proc-Type: [^\s]+', '', text)
    text = re.sub(r'Originator-Name: [^\s]+', '', text)
    text = re.sub(r'Originator-Key-Asymmetric: [^\s]+', '', text)
    text = re.sub(r'MIC-Info: [^\s]+', '', text)
    text = re.sub(r'\bwebmaster@www\.sec\.gov\b', '', text)
    text = re.sub(r'\b[\w+/=]{20,}\b', '', text)
    text = re.sub(r'\s*==\s*', '', text)
    text = re.sub(r'\.[a-zA-Z0-9]{2,5}\b', '', text)
    text = re.sub(r'\b[0-9_-]{15,}\b', '', text)
    
    # 7. 处理排版符号
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

### 测试清洗效果

In [11]:
# 测试文件路径
test_file_path = r'd:\BigFiles\hw2\annual_report_data\20161003_10-K_edgar_data_1574969_0001144204-16-126546.txt'

# 读取文件内容
with open(test_file_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

print(f'原始文本长度: {len(content)}')

# 清洗文本
cleaned_content = clean_annual_report(content)

print(f'清洗后文本长度: {len(cleaned_content)}')

# 导出清洗后的文本
export_path = r'd:\BigFiles\hw2\cleaned_test_output.txt'
with open(export_path, 'w', encoding='utf-8') as f:
    f.write(cleaned_content)
print(f'\n清洗后的文本已导出至: {export_path}')

原始文本长度: 388209
清洗后文本长度: 374097

清洗后的文本已导出至: d:\BigFiles\hw2\cleaned_test_output.txt


### 批量处理所有年报文件

In [12]:
# 批量处理所有年报文件
data_dir = r'd:\BigFiles\hw2\annual_report_data'
output_dir = r'd:\BigFiles\hw2\cleaned_annual_reports'

# 创建输出目录
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 处理所有文件
for filename in os.listdir(data_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(data_dir, filename)
        output_path = os.path.join(output_dir, f'cleaned_{filename}')
        
        # 读取文件
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # 清洗
        cleaned_content = clean_annual_report(content)
        
        # 保存清洗后的文件
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(cleaned_content)
        
        print(f'处理完成: {filename}')

print('所有文件处理完成！')

处理完成: 19970321_10-k_edgar_data_874213_0001011243-97-000002_1.txt
处理完成: 19970828_10-k_edgar_data_937598_0001009448-97-000055_1.txt
处理完成: 20020531_10-k_edgar_data_730255_0000730255-02-000014_1.txt
处理完成: 20070830_10-K_edgar_data_216039_0000950137-07-013341.txt
处理完成: 20070830_10-K_edgar_data_718916_0001368775-07-000031.txt
处理完成: 20070830_10-K_edgar_data_779152_0000926236-07-000089.txt
处理完成: 20100315_10-k_edgar_data_314606_0001047469-10-002218_1.txt
处理完成: 20130701_10-K_edgar_data_1289636_0001038838-13-000261.txt
处理完成: 20130701_10-K_edgar_data_317788_0001193125-13-279804.txt
处理完成: 20130701_10-K_edgar_data_924805_0001038838-13-000260.txt
处理完成: 20161003_10-K_edgar_data_1510963_0001510963-16-000037.txt
处理完成: 20161003_10-K_edgar_data_1574969_0001144204-16-126546.txt
处理完成: 20161003_10-K_edgar_data_1634424_0001634424-16-000004.txt
处理完成: 20170323_10-k_edgar_data_1066605_0001066605-17-000007_1.txt
处理完成: 20210225_10-k_edgar_data_912766_0001628280-21-003163.txt
处理完成: 20210319_10-k_edgar_data_769397_00

### 提取关键章节

尝试一

In [13]:
from pathlib import Path

# 输入/输出目录
src_dir = Path("annual_report_data")
out1_dir = Path("item1")
out7_dir = Path("item7")
out1_dir.mkdir(exist_ok=True)
out7_dir.mkdir(exist_ok=True)

# 通用段落边界：下一 Item
item_boundary_pattern = re.compile(r"""
    ^\s*                  # 行首空白
    (?:item|item\.)\s*    # item 或 item.
    (?P<num>[0-9]+[A-Za-z]?)  # 1, 1A, 7, 7A, ...
    (?:\s*[\.\-:]*)?      # 可选 . - :
    \s+                   # 至少一个空白
    """, re.I | re.X | re.M)

def capture_item_text(text, item_num):
    # 查 Item 标题区（可能包含 1、1.、1A、1.、ITEM 1 等）
    item_start = re.search(
        rf"^\s*(?:item|item\.)\s*{item_num}\b.*?(?:business|business\W|business\W+|management['’]?s discussion and analysis|md&a|m&d&a)",
        text,
        flags=re.I | re.M,
    )
    if not item_start:
        # 兼容 Item 7A 等，如果要取 Item7 主块，到 Item 8 前
        item_start = re.search(
                rf"^\s*(?:item|item\.)\s*{item_num}\b",
                text,
                flags=re.I | re.M,
        )
    if not item_start:
        return None

    start_pos = item_start.start()

    # 从本 Item 后找到下一个 Item 标题
    next_item = item_boundary_pattern.search(text, pos=item_start.end())
    end_pos = next_item.start() if next_item else len(text)

    return text[start_pos:end_pos].strip()

def normalize_text(txt):
    # 统一换行 + 去 \r
    return txt.replace("\r\n", "\n").replace("\r", "\n")

for src_path in sorted(src_dir.glob("*.txt")):
    t = normalize_text(src_path.read_text(encoding="utf-8", errors="ignore"))
    item1_text = capture_item_text(t, "1")
    item7_text = capture_item_text(t, "7")

    if item1_text:
        (out1_dir / src_path.name).write_text(item1_text, encoding="utf-8", errors="ignore")
    else:
        print(f"未匹配 Item 1: {src_path.name}")

    if item7_text:
        (out7_dir / src_path.name).write_text(item7_text, encoding="utf-8", errors="ignore")
    else:
        print(f"未匹配 Item 7: {src_path.name}")

print("提取完成。")

未匹配 Item 1: 20220311_10-k_edgar_data_1133869_0001558370-22-003341.txt
未匹配 Item 1: 20220701_10-K_edgar_data_1766016_0001477932-22-004810.txt
未匹配 Item 7: 20220701_10-K_edgar_data_1766016_0001477932-22-004810.txt
未匹配 Item 1: 20230713_10-K_edgar_data_1711012_0001410578-23-001474.txt
未匹配 Item 7: 20230713_10-K_edgar_data_1711012_0001410578-23-001474.txt
提取完成。


除了有一些文本没有匹配到，还存在匹配错误的问题。由于当前的正则表达式允许标题元素不在同一行，于是目录也被匹配进来。

尝试二

In [14]:
from pathlib import Path

# 输入/输出目录
src_dir = Path("annual_report_data")
out1_dir = Path("item1")
out7_dir = Path("item7")
out1_dir.mkdir(exist_ok=True)
out7_dir.mkdir(exist_ok=True)

# 存储未匹配的文件名
unmatched_files = []

# 通用段落边界：下一 Item
item_boundary_pattern = re.compile(r"""
    ^[ \t]*                  # 行首空白（不跨行）
    (?:item|item\.)[ \t]*    # item 或 item.
    (?P<num>[0-9]+[A-Za-z]?)  # 1, 1A, 7, 7A, ...
    (?:[ \t]*[\.\-:]*)?      # 可选 . - :
    [ \t]+                   # 至少一个空白
    """, re.I | re.X | re.M)

def capture_item_text(text, item_num):
    # 标题必须在同一行，且仅要求标题前有空行
    title_keywords = {
        "1": r"business",
        "7": r"management['’]?[ \t]+discussion[ \t]+and[ \t]+analysis|md[ \t]*&[ \t]*a|m[ \t]*&[ \t]*d[ \t]*&[ \t]*a",
    }.get(item_num, r"")

    item_start = re.search(
        rf"(?:^|\n[ \t]*\n)(?P<title>[ \t]*(?:item|item\.)[ \t]*{item_num}\b[^\n]*?(?:{title_keywords})[^\n]*)",
        text,
        flags=re.I,
    )
    if not item_start:
        # 兜底：仍要求同一行 + 仅前置空行，仅放宽关键词限制
        item_start = re.search(
            rf"(?:^|\n[ \t]*\n)(?P<title>[ \t]*(?:item|item\.)[ \t]*{item_num}\b[^\n]*)",
            text,
            flags=re.I,
        )
    if not item_start:
        return None

    start_pos = item_start.start("title")

    # 从本 Item 后找到下一个 Item 标题
    next_item = item_boundary_pattern.search(text, pos=item_start.end("title"))
    end_pos = next_item.start() if next_item else len(text)

    return text[start_pos:end_pos].strip()

def normalize_text(txt):
    # 统一换行 + 去 \r
    return txt.replace("\r\n", "\n").replace("\r", "\n")

for src_path in sorted(src_dir.glob("*.txt")):
    t = normalize_text(src_path.read_text(encoding="utf-8", errors="ignore"))
    item1_text = capture_item_text(t, "1")
    item7_text = capture_item_text(t, "7")

    if item1_text:
        (out1_dir / src_path.name).write_text(item1_text, encoding="utf-8", errors="ignore")
    else:
        print(f"未匹配 Item 1: {src_path.name}")
        unmatched_files.append({"filename": src_path.name, "missing_item": "Item 1"})

    if item7_text:
        (out7_dir / src_path.name).write_text(item7_text, encoding="utf-8", errors="ignore")
    else:
        print(f"未匹配 Item 7: {src_path.name}")
        unmatched_files.append({"filename": src_path.name, "missing_item": "Item 7"})

print("\n提取完成。")
print(f"\n未匹配文件总数: {len(unmatched_files)}")
print(f"\n未匹配文件列表已存储在变量 'unmatched_files' 中")

未匹配 Item 1: 20070830_10-K_edgar_data_718916_0001368775-07-000031.txt
未匹配 Item 1: 20220311_10-k_edgar_data_1133869_0001558370-22-003341.txt
未匹配 Item 7: 20220311_10-k_edgar_data_1133869_0001558370-22-003341.txt
未匹配 Item 1: 20220701_10-K_edgar_data_1766016_0001477932-22-004810.txt
未匹配 Item 7: 20220701_10-K_edgar_data_1766016_0001477932-22-004810.txt
未匹配 Item 1: 20230713_10-K_edgar_data_1711012_0001410578-23-001474.txt
未匹配 Item 7: 20230713_10-K_edgar_data_1711012_0001410578-23-001474.txt

提取完成。

未匹配文件总数: 7

未匹配文件列表已存储在变量 'unmatched_files' 中


观察了一下上述匹配不到的文件，匹配失败均是由于文件格式错乱，使得我们的“标题必须在同一行”和“前面必须有空行”的假设不成立。对于这部分文件我们单独处理，其余文件的匹配可以使用上面的代码。处理的思路是：
- 先删除格式，只保留文本内容。
- 然后依据前后文语义关系，进行匹配。

尝试三：单独处理未匹配文件


In [15]:
from pathlib import Path
import re

# 未匹配文件处理输出目录
unmatched_out1_dir = Path("unmatched_item1")
unmatched_out7_dir = Path("unmatched_item7")
unmatched_out1_dir.mkdir(exist_ok=True)
unmatched_out7_dir.mkdir(exist_ok=True)

def clean_annual_report(text):
    # 1. 移除标记语言标签（HTML标签）
    text = re.sub(r'<[^>]+>', '', text)
    
    # 2. 移除控制和特殊字符
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    
    # 3. 移除系统生成内容
    # 3.1 移除技术元数据
    text = re.sub(r'Proc-Type: [^\s]+', '', text)
    text = re.sub(r'Originator-Name: [^\s]+', '', text)
    text = re.sub(r'Originator-Key-Asymmetric: [^\s]+', '', text)
    text = re.sub(r'MIC-Info: [^\s]+', '', text)
    
    # 3.2 移除加密字符串和元数据残留
    text = re.sub(r'\bwebmaster@www\.sec\.gov\b', '', text)
    text = re.sub(r'\b[\w+/=]{20,}\b', '', text)
    text = re.sub(r'\s*==\s*', '', text)
    
    # 3.3 移除文件扩展名和文件名
    text = re.sub(r'\.[a-zA-Z0-9]{2,5}\b', '', text)
    text = re.sub(r'\b[0-9_-]{15,}\b', '', text)
    
    # 4. 处理排版符号
    text = re.sub(r'\s+', ' ', text)
    
    # 5. 去除首尾空白
    text = text.strip()
    
    return text

def is_directory_content(text):
    """
    判断是否为目录内容
    判断规则：
    1. 若在匹配内容后的 150 个字符内出现 3 个及以上的'item'关键词，则判定为目录
    2. 若在匹配内容后的 150 个字符内检测到连续 4 个及以上的"."字符，则判定为目录
    """
    if len(text) > 150:
        check_text = text[:150]
    else:
        check_text = text
    
    # 规则 1: 检查'item'关键词数量
    item_count = len(re.findall(r'\bitem\b', check_text, re.I))
    if item_count >= 3:
        return True
    
    # 规则 2: 检查连续 4 个及以上的"."字符
    if re.search(r'\.{4,}', check_text):
        return True
    
    return False

def extract_item_from_unmatched(text, item_num):
    """
    从清洗后的文本中提取指定Item内容
    匹配规则：
    - 匹配包含'item 1. business'和'item 7. management'的文本段落
    - 匹配范围截止到'item 2. properties'或'item 8. financial'出现的位置
    - 全文匹配，不只匹配第一个结果
    """
    patterns = {
        "1": {
            "start": r'item\s*1\.?\s*business',
            "end": r'item\s*2\.?\s*properties'
        },
        "7": {
            "start": r'item\s*7\.?\s*management',
            "end": r'item\s*8\.?\s*financial'
        }
    }
    
    if item_num not in patterns:
        return None
    
    start_pattern = patterns[item_num]["start"]
    end_pattern = patterns[item_num]["end"]
    
    matches = []
    
    for match in re.finditer(start_pattern, text, re.I):
        start_pos = match.start()
        
        end_match = re.search(end_pattern, text[start_pos:], re.I)
        if end_match:
            end_pos = start_pos + end_match.start()
        else:
            end_pos = len(text)
        
        extracted_text = text[start_pos:end_pos].strip()
        
        if not is_directory_content(extracted_text):
            matches.append(extracted_text)
    
    if matches:
        return "\n\n".join(matches)
    return None

def normalize_text_for_unmatched(txt):
    """
    统一换行 + 去 \r
    """
    return txt.replace("\r\n", "\n").replace("\r", "\n")

# 处理未匹配的文件
processed_count = 0
for item in unmatched_files:
    filename = item["filename"]
    missing_item = item["missing_item"]
    
    src_path = Path("annual_report_data") / filename
    
    if not src_path.exists():
        continue
    
    try:
        # 读取原始文本
        raw_text = src_path.read_text(encoding="utf-8", errors="ignore")
        
        # 数据预处理：清洗文本
        cleaned_text = clean_annual_report(raw_text)
        
        # 规范化文本
        normalized_text = normalize_text_for_unmatched(cleaned_text)
        
        # 根据缺失的Item进行提取
        if missing_item == "Item 1":
            extracted_text = extract_item_from_unmatched(normalized_text, "1")
            if extracted_text:
                (unmatched_out1_dir / filename).write_text(
                    extracted_text, encoding="utf-8", errors="ignore"
                )
                processed_count += 1
        elif missing_item == "Item 7":
            extracted_text = extract_item_from_unmatched(normalized_text, "7")
            if extracted_text:
                (unmatched_out7_dir / filename).write_text(
                    extracted_text, encoding="utf-8", errors="ignore"
                )
                processed_count += 1
                
    except Exception as e:
        print(f"处理文件 {filename} 时出错: {str(e)}")
        continue

print(f"\n未匹配文件处理完成！")
print(f"成功处理并保存的文件数量: {processed_count}")


未匹配文件处理完成！
成功处理并保存的文件数量: 7


我发现用上述方法匹配得更加准确，所以我打算不只把上述方法用于未匹配文件，而是用于所有文件。

尝试四


In [16]:
from pathlib import Path
import re

# 输入/输出目录
src_dir = Path("annual_report_data")
out1_dir = Path("item1")
out7_dir = Path("item7")
out1_dir.mkdir(exist_ok=True)
out7_dir.mkdir(exist_ok=True)

def clean_annual_report(text):
    # 1. 移除标记语言标签（HTML标签）
    text = re.sub(r'<[^>]+>', '', text)
    
    # 2. 移除控制和特殊字符
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    
    # 3. 移除系统生成内容
    # 3.1 移除技术元数据
    text = re.sub(r'Proc-Type: [^\s]+', '', text)
    text = re.sub(r'Originator-Name: [^\s]+', '', text)
    text = re.sub(r'Originator-Key-Asymmetric: [^\s]+', '', text)
    text = re.sub(r'MIC-Info: [^\s]+', '', text)
    
    # 3.2 移除加密字符串和元数据残留
    text = re.sub(r'\bwebmaster@www\.sec\.gov\b', '', text)
    text = re.sub(r'\b[\w+/=]{20,}\b', '', text)
    text = re.sub(r'\s*==\s*', '', text)
    
    # 3.3 移除文件扩展名和文件名
    text = re.sub(r'\.[a-zA-Z0-9]{2,5}\b', '', text)
    text = re.sub(r'\b[0-9_-]{15,}\b', '', text)
    
    # 4. 处理排版符号
    text = re.sub(r'\s+', ' ', text)
    
    # 5. 去除首尾空白
    text = text.strip()
    
    return text

def is_directory_content(text):
    """
    判断是否为目录内容
    判断规则：
    1. 若在匹配内容后的 150 个字符内出现 3 个及以上的'item'关键词，则判定为目录
    2. 若在匹配内容后的 150 个字符内检测到连续 4 个及以上的"."字符，则判定为目录
    """
    if len(text) > 150:
        check_text = text[:150]
    else:
        check_text = text
    
    # 规则 1: 检查'item'关键词数量
    item_count = len(re.findall(r'\bitem\b', check_text, re.I))
    if item_count >= 3:
        return True
    
    # 规则 2: 检查连续 4 个及以上的"."字符
    if re.search(r'\.{4,}', check_text):
        return True
    
    return False

def extract_item_from_text(text, item_num):
    """
    从清洗后的文本中提取指定Item内容
    匹配规则：
    - 匹配包含'item 1. business'和'item 7. management'的文本段落
    - 匹配范围截止到'item 2. properties'或'item 8. financial'出现的位置
    - 全文匹配，不只匹配第一个结果
    """
    patterns = {
        "1": {
            "start": r'item\s*1\.?\s*business',
            "end": r'item\s*2\.?\s*properties'
        },
        "7": {
            "start": r'item\s*7\.?\s*management',
            "end": r'item\s*8\.?\s*financial'
        }
    }
    
    if item_num not in patterns:
        return None
    
    start_pattern = patterns[item_num]["start"]
    end_pattern = patterns[item_num]["end"]
    
    matches = []
    
    for match in re.finditer(start_pattern, text, re.I):
        start_pos = match.start()
        
        end_match = re.search(end_pattern, text[start_pos:], re.I)
        if end_match:
            end_pos = start_pos + end_match.start()
        else:
            end_pos = len(text)
        
        extracted_text = text[start_pos:end_pos].strip()
        
        if not is_directory_content(extracted_text):
            matches.append(extracted_text)
    
    if matches:
        return "\n\n".join(matches)
    return None

def normalize_text_for_extraction(txt):
    """
    统一换行 + 去 \r
    """
    return txt.replace("\r\n", "\n").replace("\r", "\n")

# 处理所有文件
total_files = 0
item1_extracted = 0
item7_extracted = 0

for src_path in sorted(src_dir.glob("*.txt")):
    total_files += 1
    
    try:
        # 读取原始文本
        raw_text = src_path.read_text(encoding="utf-8", errors="ignore")
        
        # 数据预处理：清洗文本
        cleaned_text = clean_annual_report(raw_text)
        
        # 规范化文本
        normalized_text = normalize_text_for_extraction(cleaned_text)
        
        # 提取 Item 1
        item1_text = extract_item_from_text(normalized_text, "1")
        if item1_text:
            (out1_dir / src_path.name).write_text(item1_text, encoding="utf-8", errors="ignore")
            item1_extracted += 1
        
        # 提取 Item 7
        item7_text = extract_item_from_text(normalized_text, "7")
        if item7_text:
            (out7_dir / src_path.name).write_text(item7_text, encoding="utf-8", errors="ignore")
            item7_extracted += 1
            
    except Exception as e:
        print(f"处理文件 {src_path.name} 时出错: {str(e)}")
        continue
    
    if total_files % 5 == 0:
        print(f"已处理 {total_files} 个文件...")

print(f"\n所有文件处理完成！")
print(f"总文件数: {total_files}")
print(f"成功提取 Item 1: {item1_extracted} 个文件")
print(f"成功提取 Item 7: {item7_extracted} 个文件")

已处理 5 个文件...
已处理 10 个文件...
已处理 15 个文件...
已处理 20 个文件...
已处理 25 个文件...

所有文件处理完成！
总文件数: 25
成功提取 Item 1: 24 个文件
成功提取 Item 7: 24 个文件


## 3. 总结部分

通过对年报文本的清洗分析，我们发现了文本中存在的多种问题，如HTML标签、控制字符、系统生成内容等。这些问题对文本分析造成了干扰，因此我们需要进行有效的清洗处理。

在清理方法上，我也总结出了一些经验：
 - 去除格式之后再用正则表达式匹配，可以更准确地提取出文本中的有效信息。虽然保留格式可以利用结构特点（比如标题前后有空行）来快速匹配，但一旦结构被破坏，匹配结果就会受到影响。毕竟格式乱有各种乱法，归一化的情况只有一种。
 - 充分利用字数信息来做筛选，比如我使用“大于等于500个字符”来避免误删正文内容，以及“150个字符里出现3个及以上的"item"”来检测目录。
 - 一些文字特征也可以使用。如果光使用"item 1"来提取章节，很可能提取到正文中的引用。好在"item 1"后面必然跟的是"business"，这样我就可以准确识别标题。
 - 可以设置一个兜底规则，放宽约束。前面的规则用于严格匹配，兜底规则用于处理一些异常情况。这样可以优先保证准确率，同时也能兼顾鲁棒性。